# Análise do Tesouro Direto

Notebook que roda o fluxo do projeto seguindo o SEMMA e registra a exploração e a interpretação dos resultados. Funciona no Colab e localmente.

In [1]:
# No Colab: clona o repositório e instala as dependências. Localmente, pula esse bloco.
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('bi-tesouro-direto'):
    !git clone https://github.com/LeviJesus/bi-tesouro-direto.git
    %cd bi-tesouro-direto
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

Cloning into 'bi-tesouro-direto'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 52 (delta 17), reused 52 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 359.49 KiB | 6.78 MiB/s, done.
Resolving deltas: 100% (17/17), done.
/content/bi-tesouro-direto


## Sample: coleta

In [2]:
from src import coleta
coleta.coletar()

Baixando https://www.tesourotransparente.gov.br/ckan/dataset/df56aa42-484a-4a59-8184-7676580c81e3/resource/796d2059-14e9-44e3-80c9-2d9e30b405c1/download/PrecoTaxaTesouroDireto.csv
Salvo em /content/bi-tesouro-direto/data/raw/PrecoTaxaTesouroDireto.csv (14.7 MB)
Série 432 (selic_meta): 2191 registros
Série 433 (ipca_mensal): 72 registros


## Explore: qualidade e estatística descritiva dos dados brutos

In [3]:
import pandas as pd
bruto = pd.read_csv('data/raw/PrecoTaxaTesouroDireto.csv', sep=';', decimal=',')
print(bruto.shape)
bruto.head()

(179000, 8)


,Tipo Titulo,Data Vencimento,Data Base,Taxa Compra Manha,Taxa Venda Manha,PU Compra Manha,PU Venda Manha,PU Base Manha
0,Tesouro Selic,01/03/2027,25/11/2026,0.03,0.04,20374.84,20363.87,20363.87
1,Tesouro Selic,01/03/2031,25/11/2026,0.11,0.12,20281.78,20262.76,20262.76
2,Tesouro Selic,01/03/2028,25/11/2026,0.04,0.05,20366.42,20353.42,20353.42
3,Tesouro Selic,01/03/2029,25/11/2026,0.06,0.07,20351.18,20336.17,20336.17
4,Tesouro Prefixado,01/01/2031,25/11/2026,15.18,15.30,562.79,560.10,560.10


In [4]:
print(bruto.isna().sum())
bruto['Tipo Titulo'].value_counts()

Tipo Titulo          0
Data Vencimento      0
Data Base            0
Taxa Compra Manha    0
Taxa Venda Manha     0
PU Compra Manha      0
PU Venda Manha       0
PU Base Manha        0
dtype: int64


,count
Tipo Titulo,
Tesouro IPCA+ com Juros Semestrais,44370
Tesouro Prefixado,28350
Tesouro Prefixado com Juros Semestrais,27535
Tesouro Selic,21573
Tesouro IPCA+,19104
Tesouro IGPM+ com Juros Semestrais,15840
Tesouro Educa+,14604
Tesouro Renda+ Aposentadoria Extra,7624


In [5]:
bruto[['Taxa Compra Manha','Taxa Venda Manha','PU Compra Manha','PU Venda Manha']].describe()

,Taxa Compra Manha,Taxa Venda Manha,PU Compra Manha,PU Venda Manha
count,179000.000000,179000.000000,179000.000000,179000.000000
mean,7.182543,7.264296,2703.438847,2689.728842
std,4.150550,4.157111,2952.418931,2948.703038
min,-3.530000,-3.410000,0.000000,0.000000
25%,5.280000,5.390000,957.220000,954.860000
50%,6.840000,6.930000,1769.105000,1755.290000
75%,9.920000,10.000000,3353.337500,3333.020000
max,20.100000,20.130000,20374.840000,20363.870000


## Modify: limpeza e data warehouse

In [6]:
from src import limpeza, warehouse
tesouro, macro = limpeza.limpar()
warehouse.construir_dw(tesouro, macro)

Tesouro: 179000 linhas brutas, 59944 na janela, 59943 após limpeza
DW criado em /content/bi-tesouro-direto/data/processed/tesouro_dw.duckdb


PosixPath('/content/bi-tesouro-direto/data/processed/tesouro_dw.duckdb')

In [7]:
con = warehouse.abrir_dw()
con.execute('''
    SELECT d.tipo_titulo, count(distinct d.id_titulo) AS vencimentos, count(*) AS observacoes,
           round(avg(f.taxa_venda), 2) AS taxa_media
    FROM fato_titulo_dia f JOIN dim_titulo d USING (id_titulo)
    GROUP BY 1 ORDER BY 1
''').df()

,tipo_titulo,vencimentos,observacoes,taxa_media
0,Tesouro Educa+,19,14604,6.98
1,Tesouro IGPM+ com Juros Semestrais,1,1248,6.78
2,Tesouro IPCA+,8,6430,6.92
3,Tesouro IPCA+ com Juros Semestrais,11,11190,6.74
4,Tesouro Prefixado,10,6239,12.79
5,Tesouro Prefixado com Juros Semestrais,8,6882,12.78
6,Tesouro Renda+ Aposentadoria Extra,8,7624,6.70
7,Tesouro Selic,8,5726,0.07


## Model: KPIs

In [8]:
from src import metricas
kpis = metricas.calcular_kpis()
metricas.salvar_kpis(kpis)
kpis[['nome_titulo','taxa_atual','rentabilidade_acum','vol_taxa','spread_selic','spread_real_ipca','drawdown_max','duration_anos']].round(2)

,nome_titulo,taxa_atual,rentabilidade_acum,vol_taxa,spread_selic,spread_real_ipca,drawdown_max,duration_anos
0,Tesouro Educa+ 2030,7.41,4.79,1.07,-1.37,7.41,-11.58,4.05
1,Tesouro Educa+ 2031,7.56,29.56,1.02,-1.21,7.56,-5.65,5.05
2,Tesouro Educa+ 2032,7.76,26.00,0.99,-1.00,7.76,-7.38,6.06
3,Tesouro Educa+ 2033,7.86,22.88,0.96,-0.90,7.86,-8.82,7.06
4,Tesouro Educa+ 2034,7.91,20.00,0.94,-0.85,7.91,-10.09,8.05
5,Tesouro Educa+ 2035,7.95,17.13,0.90,-0.80,7.95,-11.25,9.05
6,Tesouro Educa+ 2036,7.95,14.62,0.87,-0.80,7.95,-12.65,10.06
7,Tesouro Educa+ 2037,7.92,12.50,0.84,-0.84,7.92,-13.85,11.06
8,Tesouro Educa+ 2038,7.88,10.51,0.81,-0.88,7.88,-14.95,12.05
9,Tesouro Educa+ 2039,7.83,8.78,0.78,-0.93,7.83,-15.94,13.05


## Model e Assess: previsão da taxa do título âncora

In [9]:
from src import preditivo
prev = preditivo.treinar_e_avaliar()
print(prev['titulo'])
pd.DataFrame(prev['metricas']).round(3)

Tesouro Prefixado 2029


,modelo,baseline
MAE,0.118,0.408
RMSE,0.231,0.576
R2,0.889,0.308


### Interpretação

**O modelo supera o baseline?** Sim. No período de teste (20% mais recente, 232 dias), a regressão linear do Tesouro Prefixado 2029 teve MAE de 0,118 p.p., RMSE de 0,231 p.p. e R² de 0,889. A média móvel de 30 dias teve MAE de 0,408, RMSE de 0,576 e R² de 0,308. O erro do modelo é cerca de um terço do erro do baseline.

**Por que a diferença é grande.** A taxa de um dia é muito parecida com a do dia anterior (o coeficiente de `lag1` é cerca de 0,92). O modelo usa essa informação, e a média de 30 dias reage devagar quando a taxa muda de patamar. Os outros coeficientes são pequenos: a tendência temporal e a Selic quase não contribuem.

**O que a projeção sugere.** Para os próximos 30 dias úteis, a taxa fica praticamente estável, indo de cerca de 15,02% para 14,97% a.a. O modelo não indica mudança de direção relevante no curto prazo.

**Limitações.**
- O teste é de previsão um passo à frente, com a taxa real do dia anterior como entrada. Isso favorece o modelo com defasagem. Um baseline "repete o valor de ontem" provavelmente ficaria bem mais perto do modelo do que a média móvel.
- A projeção de 30 dias usa as próprias previsões como defasagens, então tende a ficar perto do último valor e não antecipa choques como mudança na Selic ou no cenário fiscal.
- A Selic é mantida constante na projeção.
- Um único título e um único split temporal. Os resultados não generalizam para os demais títulos.

**Conclusão.** O modelo é útil como referência de curto prazo e supera o baseline escolhido. A decisão de alocação não deve se basear nele sozinho, mas junto com os KPIs (spread sobre a Selic, taxa real, drawdown e duration).

## Assess: dashboards

In [10]:
from src import dashboards
dashboards.gerar_dashboard(previsao=prev)

Dashboard salvo em /content/bi-tesouro-direto/docs/index.html


PosixPath('/content/bi-tesouro-direto/docs/index.html')